In [38]:
import json
from pprint import pprint

In [39]:
def filter_conversations(
    dataset,
    min_messages=None, max_messages=None,
    min_thoughts=None, max_thoughts=None,
    model_name=None, model_provider=None,
    min_age=None, max_age=None,
    gender=None, education=None,
    min_frequency=None, max_frequency=None,
):
    # `model_name`, `model_provider`, `gender`, `education` accept a string or list of strings.
    def as_set(x):
        return {x} if isinstance(x, str) else set(x) if x else None

    model_names = as_set(model_name)
    model_providers = as_set(model_provider)
    genders = as_set(gender)
    educations = as_set(education)

    def to_int(v):
        try: return int(v)
        except (TypeError, ValueError): return None

    filtered = {}
    for key, conv in dataset.items():
        msgs = conv.get("messages") or []
        n_msgs = len(msgs)
        n_thoughts = sum(len(m.get("reasons") or []) + len(m.get("reactions") or []) for m in msgs)

        if min_messages is not None and n_msgs < min_messages: continue
        if max_messages is not None and n_msgs > max_messages: continue
        if min_thoughts is not None and n_thoughts < min_thoughts: continue
        if max_thoughts is not None and n_thoughts > max_thoughts: continue
        if model_names is not None and conv.get("model_name") not in model_names: continue
        if model_providers is not None and conv.get("model_provider") not in model_providers: continue

        survey = (conv.get("survey_answers") or [{}])[0]
        age = to_int(survey.get("age"))
        freq = to_int(survey.get("frequency"))

        if min_age is not None and (age is None or age < min_age): continue
        if max_age is not None and (age is None or age > max_age): continue
        if min_frequency is not None and (freq is None or freq < min_frequency): continue
        if max_frequency is not None and (freq is None or freq > max_frequency): continue
        if genders is not None and survey.get("gender") not in genders: continue
        if educations is not None and survey.get("education") not in educations: continue

        filtered[key] = conv

    print(f"Filtered: {len(filtered):,} / {len(dataset):,} conversations")
    return filtered

In [40]:
# Option 1: load from HuggingFace
from datasets import load_dataset
ds = load_dataset("SCAI-JHU/ThoughtTrace", split="train")
data = {row["id"]: row for row in ds}

# Option 2: load from JSONL
# data = {}
# with open("ThoughtTrace.jsonl") as f:
#     for line in f:
#         row = json.loads(line)
#         data[row["id"]] = row

In [41]:
# Example 1: Anthropic/OpenAI conversations with 6-10 messages from users aged 18-40
subset1 = filter_conversations(
    data,
    min_messages=6,
    max_messages=10,
    model_provider=["Anthropic", "OpenAI"],
    min_age=18, max_age=40,
)
pprint(next(iter(subset1.values())))

Filtered: 306 / 2,155 conversations
{'created_at': '2026-03-18T15:10:28.787000+00:00',
 'id': 'user7_task1_conversation1',
 'messages': [{'content': 'hello',
               'id': 1773847728950,
               'reactions': None,
               'reasons': [],
               'timestamp': '2026-03-18T15:28:48.950Z',
               'type': 'user'},
              {'content': 'Hello! How can I help?',
               'id': 1773847730910,
               'reactions': [],
               'reasons': None,
               'timestamp': '2026-03-18T15:28:50.909Z',
               'type': 'assistant'},
              {'content': 'as an individual can you give the bank funds of '
                          'like 50 Million as a sort of savings or investment '
                          'for a yearly profit',
               'id': 1773847822596,
               'reactions': None,
               'reasons': [],
               'timestamp': '2026-03-18T15:30:22.596Z',
               'type': 'user'},
              {

In [42]:
# Example 2: frequent female users (frequency >= 4) with undergraduate/graduate education, at least 3 thoughts
subset2 = filter_conversations(
    data,
    min_thoughts=3,
    gender="Female",
    education=["Undergraduate", "Graduate"],
    min_frequency=4,
)
pprint(next(iter(subset2.values())))

Filtered: 371 / 2,155 conversations
{'created_at': '2026-03-18T15:10:55.234000+00:00',
 'id': 'user9_task1_conversation1',
 'messages': [{'content': "Hello! I'd like to have a meal plan from Monday to "
                          'Friday, vegetarian, with a maximum of 500cal each '
                          'meal, lots of veggies and a sweet snack after '
                          'dinner. Also a list of ingredients to buy',
               'id': 1773846870131,
               'reactions': None,
               'reasons': [{'content': "It's a daily (or weekly) task that I "
                                       'do not enjoy completing, this makes it '
                                       'easier to meal prep and to not buy '
                                       'unnecessary food that will end up in '
                                       'the trash',
                            'label': 'task_motivation',
                            'timestamp': '2026-03-18T15:15:58.938Z'}],
       